In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("parquet")\
        .load("abfss://bronze@databricksetestorage2003.dfs.core.windows.net/data/products")
display(df)

In [0]:
df = df.drop(col("_rescued_data"))
display(df)

In [0]:
df.createOrReplaceTempView("products")

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_ete.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE 
LANGUAGE SQL
RETURN p_price*0.90

In [0]:
%sql
select product_id, databricks_ete.bronze.discount_func(price) as discounted_price
from products

In [0]:
from pyspark.sql.functions import round
df = df.withColumn('discounted_price',round(expr("databricks_ete.bronze.discount_func(price)"), 2))
display(df)

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_ete.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS 
$$
    return p_brand.upper()
$$

In [0]:
%sql
select product_id, brand, databricks_ete.bronze.upper_func(brand) as brand_upper
from products

In [0]:
df.write.format("delta")\
    .mode("append")\
    .option("path", "abfss://silver@databricksetestorage2003.dfs.core.windows.net/products")\
    .save()